In [1]:
# 1. Montar Google Drive
from google.colab import drive
import os, subprocess

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

Mounted at /content/drive


In [2]:
# 2. Definir caminhos
ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")

BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
RAW_PATH  = os.path.join(BASE_PATH, "data_raw")
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
os.makedirs(PROC_PATH, exist_ok=True)

print("RAW_PATH :", RAW_PATH)
print("PROC_PATH:", PROC_PATH)

RAW_PATH : /content/drive/MyDrive/TCC_USP/data_raw
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [3]:
# 3. Carregar dataset de notícias reais
import pandas as pd
news_file = os.path.join(RAW_PATH, "noticias_real.csv")
assert os.path.exists(news_file), f"Arquivo não encontrado: {news_file}"

df_news = pd.read_csv(news_file)
print("Shape inicial:", df_news.shape)
display(df_news.head())

Shape inicial: (81, 4)


,data,fonte,titulo,texto
0,2025-09-27,IGN,"""Apenas 1 em cada 10 mil jogadores encontraria...",Esqueceu ou decidiu não comprar a Pena? O game...
1,2025-09-27,Sapo.pt,Google Fotos vai ao Tinder copiar uma nova fun...,O Google Fotos poderá receber uma funcionalida...
2,2025-09-27,Observador.pt,Fogo em Oleiros dominado ao início da tarde,Fogo deflagrou por volta das 11:50 deste sábad...
3,2025-09-27,Abril.com.br,"Os maiores pecados de quem bebe vinho, segundo...",Vale conferir as dicas preparadas especialment...
4,2025-09-27,Metropoles.com,Alice Wegmann reage após gafe em cena de Vale ...,Cena de Vale Tudo foi detonada nas redes socia...


In [4]:
# 4. Padronizar colunas
df_news = df_news.rename(columns={
    "data": "data",
    "fonte": "fonte",
    "titulo": "titulo",
    "texto": "texto"
})

# Garantir formato de data
df_news["data"] = pd.to_datetime(df_news["data"], errors="coerce")

In [5]:
# 5. Pré-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))

def preprocess_text(t):
    if pd.isna(t):
        return ""
    t = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Criar coluna limpa a partir de título + texto
df_news["texto_completo"] = (df_news["titulo"].fillna("") + " " + df_news["texto"].fillna("")).str.strip()
df_news["clean_text"] = df_news["texto_completo"].apply(preprocess_text)

print("✅ Pré-processamento concluído!")
display(df_news[["data","fonte","titulo","clean_text"]].head())

✅ Pré-processamento concluído!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,data,fonte,titulo,clean_text
0,2025-09-27,IGN,"""Apenas 1 em cada 10 mil jogadores encontraria...",apenas cada mil jogadores encontraria jogou si...
1,2025-09-27,Sapo.pt,Google Fotos vai ao Tinder copiar uma nova fun...,google fotos vai tinder copiar nova funcionali...
2,2025-09-27,Observador.pt,Fogo em Oleiros dominado ao início da tarde,fogo oleiros dominado início tarde fogo deflag...
3,2025-09-27,Abril.com.br,"Os maiores pecados de quem bebe vinho, segundo...",maiores pecados bebe vinho segundo papa francê...
4,2025-09-27,Metropoles.com,Alice Wegmann reage após gafe em cena de Vale ...,alice wegmann reage após gafe cena vale tudo d...


In [6]:
# 6. Salvar dataset processado
news_out = os.path.join(PROC_PATH, "noticias_real_clean.csv")
df_news.to_csv(news_out, index=False, encoding="utf-8-sig")

print("Arquivo salvo em:", news_out)

Arquivo salvo em: /content/drive/MyDrive/TCC_USP/data_processed/noticias_real_clean.csv
